In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, glob, os
import scipy.io as sio

## Logical flow

per patient:
* create directory to populate with only select OSort figs for faster inspection
* create preQC_df where I will manually perform QC
* for filtered neurons, incorporate spike_times, regions, coordinates for OSort .mats.
* merge as needed
* inspect & saves pt_neurs_info_df, where rows = neurons, cols = spikes, region, coords, etc. in outputs/processed_data/20{patient}/neurs_info_df.parquet

### variables

In [ ]:
suffix = ['','_max','_notch600'][2]
patient = 2605
# save = False

units_dir = f'../../outputs/unit_figs'
units_all_dir, units_waves_dir = f'{units_dir}/all/20{patient}{suffix}', f'{units_dir}/waves/20{patient}{suffix}'
units_keeps_dir,units_maybes_dir,units_questionables_dir,units_mergers_dir =\
        f'{units_dir}/keeps/20{patient}', f'{units_dir}/maybes/20{patient}', f'{units_dir}/questionables/20{patient}', f'{units_dir}/mergers/20{patient}'

for dir in [units_all_dir,units_waves_dir,units_keeps_dir,units_maybes_dir,units_questionables_dir,units_mergers_dir]:    os.makedirs(dir, exist_ok=True)
print(f'total units: {len(glob.glob(f"{units_all_dir}/*"))}')

### 1. set up preQC, populate only-cluster and only-wave fig directories

In [13]:
# parse osort figs
for file in glob.glob(f'../../data/20{patient}/osort_mat/figs{suffix}/5/*'):

    if 'CL' in os.path.basename(file) and 'ALL' not in os.path.basename(file): # clusters
        dest = os.path.join(units_all_dir, os.path.basename(file))
        if not os.path.exists(dest): os.system(f'cp {file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')

    if 'WAVES' in os.path.basename(file): # to consider mergers
        dest = os.path.join(units_waves_dir, os.path.basename(file))
        if not os.path.exists(dest): os.system(f'cp {file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')


### create preQC_df with extra columns

In [14]:
chanIDs,unitIDs = [],[]
for file in glob.glob(f'{units_all_dir}/*'):
    chanIDs.append(int(os.path.basename(file).split('_')[0][1:])); unitIDs.append(int(os.path.basename(file).split('_')[2]))
preQC_df = pd.DataFrame({'chanID': chanIDs, 'unitID': unitIDs}).sort_values(by=['chanID', 'unitID']).reset_index(drop=True)

preQC_df['keeps'], preQC_df['maybes'], preQC_df['questionables'], preQC_df['mergers'], preQC_df['notes'] = np.nan, np.nan, np.nan, np.nan, np.nan

preQC_df_path = f'../../outputs/processed_data/20{patient}/preQC_pt{patient}.csv'
os.makedirs(os.path.dirname(preQC_df_path), exist_ok=True)
if not os.path.exists(preQC_df_path): preQC_df.to_csv(preQC_df_path, index=False); print(f'Saving preQC_df to {preQC_df_path}')
else: print(f'preQC_df already exists at {preQC_df_path}, skipping save')
preQC_df

Saving preQC_df to ../../outputs/processed_data/202605/preQC_pt2605.csv


,chanID,unitID,keeps,maybes,questionables,mergers,notes


### 2. load QC; separate keeps/maybes/questionables/mergers neur figs in the data/units dir

In [5]:
QC_df = pd.read_csv(f'../../outputs/processed_data/20{patient}/QC_pt{patient}.csv')
keeps_df = QC_df[QC_df['keeps'] == 1].copy().reset_index(drop=True)
maybes_df = QC_df[~QC_df['maybes'].isna()].copy().reset_index(drop=True)
questionables_df = QC_df[QC_df['questionables'] == 1].copy().reset_index(drop=True)
mergers_df = QC_df[~QC_df['mergers'].isna()].copy().reset_index(drop=True)
# keeps + mergers
possible_neurs_info_df = pd.concat([keeps_df, mergers_df]).drop_duplicates().reset_index(drop=True)

for unit_file in glob.glob(f'{units_all_dir}/*'):
    chanID, unitID = int(os.path.basename(unit_file).split('_')[0][1:]), int(os.path.basename(unit_file).split('_')[2])

    if ((keeps_df['chanID'] == chanID) & (keeps_df['unitID'] == unitID)).any():
        dest = f'{units_keeps_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):    os.system(f'cp {unit_file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')
    if ((maybes_df['chanID'] == chanID) & (maybes_df['unitID'] == unitID)).any():
        dest = f'{units_maybes_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):    os.system(f'cp {unit_file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')
    if ((questionables_df['chanID'] == chanID) & (questionables_df['unitID'] == unitID)).any():
        dest = f'{units_questionables_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):    os.system(f'cp {unit_file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')
    if ((mergers_df['chanID'] == chanID) & (mergers_df['unitID'] == unitID)).any():
        dest = f'{units_mergers_dir}/{os.path.basename(unit_file)}'
        if not os.path.exists(dest):    os.system(f'cp {unit_file} {dest}'); print(f'Copying to {dest}')
        else: print(f'File already exists at {dest}, skipping copy')

# checks
for label, df, target_dir in [('keeps', keeps_df, units_keeps_dir), ('maybes', maybes_df, units_maybes_dir), ('mergers', mergers_df, units_mergers_dir)]:
    dir_files, df_files = set(os.path.basename(f) for f in glob.glob(f'{target_dir}/*.png')), set()
    for _, row in df.iterrows():
        matches = glob.glob(f'{units_all_dir}/A{int(row["chanID"])}_*_{int(row["unitID"])}_*.png')
        df_files.update(os.path.basename(f) for f in matches)
    extra_in_dir, missing_from_dir = dir_files-df_files, df_files-dir_files
    if extra_in_dir:   print(f'[{label}] extra files (in dir, not in df): {extra_in_dir}')
    if missing_from_dir: print(f'[{label}] missing files (in df, not in dir): {missing_from_dir}')

assert len(keeps_df) == len(glob.glob(f'{units_keeps_dir}/*.png')), f'Length mismatch: keeps_df has {len(keeps_df)} rows, but {len(glob.glob(f"{units_keeps_dir}/*.png"))} files in {units_keeps_dir}'
assert len(maybes_df) == len(glob.glob(f'{units_maybes_dir}/*.png')), f'Length mismatch: maybes_df has {len(maybes_df)} rows, but {len(glob.glob(f"{units_maybes_dir}/*.png"))} files in {units_maybes_dir}'
# assert len(mergers_df) == len(glob.glob(f'{units_mergers_dir}/*.png')), f'Length mismatch: mergers_df has {len(mergers_df)} rows, but {len(glob.glob(f"{units_mergers_dir}/*.png"))} files in {units_mergers_dir}'

possible_neurs_info_df

File already exists at ../../outputs/unit_figs/maybes/202521/A195_CL_616_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/keeps/202521/A219_CL_1329_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/questionables/202521/A219_CL_1329_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/maybes/202521/A194_CL_174_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/maybes/202521/A200_CL_340_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/keeps/202521/A224_CL_3097_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/questionables/202521/A224_CL_3097_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/maybes/202521/A197_CL_856_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/keeps/202521/A218_CL_3887_THM_1.png, skipping copy
File already exists at ../../outputs/unit_figs/mergers/202521/A218_CL_3887_THM_1.png, skipping copy
Fi

,chanID,unitID,keeps,maybes,questionables,mergers
0,216,5338,1.0,NaN,1.0,NaN
1,217,1148,1.0,NaN,NaN,NaN
2,218,3804,1.0,NaN,1.0,NaN
3,218,3845,1.0,NaN,NaN,NaN
4,218,3887,1.0,NaN,NaN,1.0
5,218,3888,1.0,NaN,NaN,1.0
6,219,1329,1.0,NaN,1.0,NaN
7,219,1709,1.0,NaN,NaN,NaN
8,220,2305,1.0,NaN,NaN,NaN
9,220,2374,1.0,NaN,NaN,NaN


### 3. create neurs_info_df for neurs in possible_neurs_info_df and taking in spike times from osort mat files

### helpers

In [ ]:
def getunitID2spikes(unitIDs, spikes, possible_neurs_info_df):
    ''' return dict with keys=unique units, and vals = list of corresponding spikes '''    
    unit2spikes = {}
    for unitID, spike in zip(unitIDs, spikes):
        if unitID not in possible_neurs_info_df['unitID'].tolist(): continue
        if unitID not in unit2spikes: unit2spikes[unitID] = [] # initialize
        unit2spikes[unitID].append(spike)

    return unit2spikes

### First, add spike data from OSort mats.

In [ ]:
samp_rate = 1000000
# columns
chanID_list, unitID_list, spikes_list, num_spikes_list, FR_list = [], [], [], [], []

# go through OSort mat files
for mat_file in glob.glob(f'../../data/20{patient}/osort_mat/sorted_mats{suffix}/5/*_sorted_new.mat'):

    chanMat, chanID = sio.loadmat(mat_file), int(os.path.basename(mat_file).split('_')[0][1:])

    if chanMat['assignedNegative'].size == 0: continue
    spike_vector = chanMat['newTimestampsNegative'][0] # #n_spikes
    unit_vector = chanMat['assignedNegative'][0] # #n_units

    # create unit_vector => [spike_vector] for QCed units
    unit2spikes = getunitID2spikes(unit_vector, spike_vector, possible_neurs_info_df)
    for unitID, unit_spike_list in unit2spikes.items():
        chanID_list.append(chanID); unitID_list.append(unitID)
        spikes = np.array(unit_spike_list) / samp_rate  # seconds
        spikes_list.append(spikes)
        num_spikes_list.append(len(spikes)); FR_list.append(len(spikes) / (spikes[-1] - spikes[0]))

pt_neurs_info_df = pd.DataFrame({'chanID': chanID_list, 'unitID': unitID_list, 'spikes': spikes_list, 'num_spikes': num_spikes_list, 'FR': FR_list})
pt_neurs_info_df = pd.merge(pt_neurs_info_df, possible_neurs_info_df[['chanID', 'unitID', 'keeps', 'mergers']], on=['chanID', 'unitID'], how='left')
pt_neurs_info_df = pt_neurs_info_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
assert len(pt_neurs_info_df) == len(possible_neurs_info_df), f'Length mismatch: pt_neurs_info_df has {len(pt_neurs_info_df)} rows, possible_neurs_info_df has {len(possible_neurs_info_df)} rows'
pt_neurs_info_df


### add region column by mapping channel -> region (label)

In [ ]:
chanMap = sio.loadmat(glob.glob(f'../../data/20{patient}/records/*ChannelMap*.mat')[0])
channelMap = chanMap['ChannelMap2'].flatten() if patient == 2518 else chanMap['ChannelMap1'].flatten()
labelMap = chanMap['LabelMap'].flatten(); labelMap = np.array([str(label.squeeze()) for label in labelMap])

# patient 21's spike data skips a bank of 8 channels above 208; shift back to match the channel map
if patient == 2521: pt_neurs_info_df['chanID'] = np.where(pt_neurs_info_df['chanID'] >= 209, pt_neurs_info_df['chanID'] - 8, pt_neurs_info_df['chanID'])

channel2label = dict(zip(channelMap[~np.isnan(channelMap)], labelMap[~np.isnan(channelMap)]))
pt_neurs_info_df['region'] = pt_neurs_info_df['chanID'].map(channel2label)
assert pt_neurs_info_df['region'].notna().all(), f'unmapped channels: {pt_neurs_info_df.loc[pt_neurs_info_df["region"].isna(), "chanID"].tolist()}'

pt_neurs_info_df

### add coordinates columns by mapping region->coords

In [9]:
def clean_entry(x): # cleaning function to handle nested arrays and bytes
    while isinstance(x, (np.ndarray, list)):    x = x[0]
    if isinstance(x, (bytes, bytearray)):    x = x.decode("utf-8", errors="ignore")
    return str(x)

In [10]:
electrodeInfo = sio.loadmat(glob.glob(f'../../data/20{patient}/records/*DI_Electrodes*.mat')[0])

ElecMapRaw   = pd.DataFrame(electrodeInfo['ElecMapRaw']); region_s = ElecMapRaw[0].apply(clean_entry)
region2id_df = pd.DataFrame({"region":region_s.values, "ID":np.arange(len(region_s))})

ElecXYZRaw   = pd.DataFrame(electrodeInfo['ElecXYZRaw']) # ID -> coordinates
id2xyz_df = ElecXYZRaw.reset_index().rename(columns={'index':'ID', 0:'x', 1:'y', 2:'z'})

# ElecAtlasRaw = pd.DataFrame(electrodeInfo['ElecAtlasRaw']) # atlas coords?
# atlas_index = 0; atlas_s  = ElecAtlasRaw.iloc[:, atlas_index].apply(clean_entry)  # Series of atlas regions
# xyz2atlasRegions = pd.DataFrame({"ID": np.arange(len(atlas_s)),"atlas_region": atlas_s.values})

pt_neurs_info_df = (pt_neurs_info_df.merge(region2id_df, on='region', how='left').merge(id2xyz_df, on='ID', how='left')) # .merge(xyz2atlasRegions, on='ID', how='left')
pt_neurs_info_df = pt_neurs_info_df.drop(columns=['ID']); pt_neurs_info_df = pt_neurs_info_df.sort_values(by=['chanID', 'unitID']).reset_index(drop=True)
pt_neurs_info_df

,chanID,unitID,spikes,num_spikes,FR,keeps,mergers,region,region_parsed,x,y,z
0,208,5338,"[2.5876333333333337, 2.6418, 2.674966666666667...",7409,4.273464,1.0,NaN,mRACC8,ACC,6.600005,39.867826,17.599995
1,209,1148,"[34.2711, 34.276633333333336, 34.3298, 36.1802...",1067,0.644605,1.0,NaN,mRAHIP1,HPC,19.800004,1.098719,-15.600003
2,210,3804,"[309.8436333333334, 317.89003333333335, 448.75...",1564,1.069608,1.0,NaN,mRAHIP2,HPC,21.000004,1.098719,-15.600003
3,210,3845,"[186.2033, 189.2548666666667, 189.284366666666...",3452,2.174044,1.0,NaN,mRAHIP2,HPC,21.000004,1.098719,-15.600003
4,210,3887,"[0.34526666666666667, 0.3496, 0.35416666666666...",15705,8.859160,1.0,1.0,mRAHIP2,HPC,21.000004,1.098719,-15.600003
5,210,3888,"[0.3418666666666667, 0.6712, 1.758166666666666...",3976,2.241688,1.0,1.0,mRAHIP2,HPC,21.000004,1.098719,-15.600003
6,211,1329,"[39.68333333333334, 42.36246666666667, 42.5754...",1708,1.006507,1.0,NaN,mRAHIP3,HPC,18.600004,1.098719,-15.600003
7,211,1709,"[5.8342, 33.16630000000001, 34.1727, 34.288, 3...",2786,1.606023,1.0,NaN,mRAHIP3,HPC,18.600004,1.098719,-15.600003
8,212,2305,"[34.1908, 34.4417, 34.516533333333335, 34.5623...",13423,7.790637,1.0,NaN,mRAHIP4,HPC,18.600004,-0.100325,-15.600003
9,212,2374,"[10.1639, 32.283433333333335, 35.6210666666666...",710,0.406395,1.0,NaN,mRAHIP4,HPC,18.600004,-0.100325,-15.600003


### 4. Merge clusters as needed

In [11]:
merged_rows = []; merge_mask = pt_neurs_info_df['mergers'].notna()

# extract merger rows as df
for _, merge_group in pt_neurs_info_df[merge_mask].groupby('mergers'):
    merged_spikes = np.sort(np.concatenate(merge_group['spikes'].values))

    first_row = merge_group.iloc[0] # populates chanID, region, merge_cluster, x, y, z
    merged_rows.append({
        **{col: first_row[col] for col in pt_neurs_info_df.columns},
        'unitID':'/'.join(merge_group['unitID'].astype(str).tolist()), 'spikes':merged_spikes,
        'num_spikes': len(merged_spikes), 'FR': len(merged_spikes) / (merged_spikes[-1] - merged_spikes[0]),
        'keeps':      1,
    })

# concat original df without merged rows + new merged rows
pt_neurs_info_df = pd.concat([pt_neurs_info_df[~merge_mask], pd.DataFrame(merged_rows)], ignore_index=True); pt_neurs_info_df = pt_neurs_info_df.sort_values(by=['chanID']).reset_index(drop=True)
pt_neurs_info_df

,chanID,unitID,spikes,num_spikes,FR,keeps,mergers,region,region_parsed,x,y,z
0,208,5338,"[2.5876333333333337, 2.6418, 2.674966666666667...",7409,4.273464,1.0,NaN,mRACC8,ACC,6.600005,39.867826,17.599995
1,209,1148,"[34.2711, 34.276633333333336, 34.3298, 36.1802...",1067,0.644605,1.0,NaN,mRAHIP1,HPC,19.800004,1.098719,-15.600003
2,210,3804,"[309.8436333333334, 317.89003333333335, 448.75...",1564,1.069608,1.0,NaN,mRAHIP2,HPC,21.000004,1.098719,-15.600003
3,210,3845,"[186.2033, 189.2548666666667, 189.284366666666...",3452,2.174044,1.0,NaN,mRAHIP2,HPC,21.000004,1.098719,-15.600003
4,210,3887/3888,"[0.3418666666666667, 0.34526666666666667, 0.34...",19681,11.096242,1.0,1.0,mRAHIP2,HPC,21.000004,1.098719,-15.600003
5,211,1329,"[39.68333333333334, 42.36246666666667, 42.5754...",1708,1.006507,1.0,NaN,mRAHIP3,HPC,18.600004,1.098719,-15.600003
6,211,1709,"[5.8342, 33.16630000000001, 34.1727, 34.288, 3...",2786,1.606023,1.0,NaN,mRAHIP3,HPC,18.600004,1.098719,-15.600003
7,212,2305,"[34.1908, 34.4417, 34.516533333333335, 34.5623...",13423,7.790637,1.0,NaN,mRAHIP4,HPC,18.600004,-0.100325,-15.600003
8,212,2374,"[10.1639, 32.283433333333335, 35.6210666666666...",710,0.406395,1.0,NaN,mRAHIP4,HPC,18.600004,-0.100325,-15.600003
9,214,878,"[0.05560000000000001, 0.7493333333333334, 0.82...",3460,1.949670,1.0,NaN,mRAHIP6,HPC,19.800004,-1.299370,-15.600003


### 5. inspect, typecast, & save

In [12]:
pt_neurs_info_df['spikes'] = pt_neurs_info_df['spikes'].apply(lambda x: np.array(x))
print('example neuron')
print(f'last 5 spikes (s): {pt_neurs_info_df["spikes"].iloc[0][-5:]}\nlast 5 spikes (min): {pt_neurs_info_df["spikes"].iloc[0][-5:]/60}')
pt_neurs_info_df['unitID'] = pt_neurs_info_df['unitID'].astype(str)
if 'patient' not in pt_neurs_info_df: pt_neurs_info_df.insert(0, 'patient', patient)

processed_data_dir = f'../../outputs/processed_data/20{patient}'; os.makedirs(processed_data_dir, exist_ok=True)
parquet_path = os.path.join(processed_data_dir, 'neurs_info_df.parquet')
if not os.path.exists(parquet_path):    pt_neurs_info_df.to_parquet(parquet_path, index=False); print('saving pt_neurs_info_df')
else: print(f'pt_neurs_info_df already exists at {parquet_path}, skipping save')    


example neuron
last 5 spikes (s): [1733.50833333 1733.69243333 1733.75293333 1734.59233333 1736.31      ]
last 5 spikes (min): [28.89180556 28.89487389 28.89588222 28.90987222 28.9385    ]
pt_neur_df already exists at ../../outputs/processed_data/202521/neurs_df.parquet, skipping save
